In [5]:
import torch
from einops import rearrange, einsum
import einx

# Basic equivalence to Y = D @ A.T
# D has shape: batch sequence d_in (e.g., B T d_in)
# A has shape: d_out d_in
# Einsum: D A -> Y, reduces over d_in and yields ... d_out
# Example patterns (not executed here):
# Y = einsum(D, A, "batch sequence d_in, d_out d_in -> batch sequence d_out")
# Y = einsum(D, A, "... d_in, d_out d_in -> ... d_out")


In [ ]:
# Minimal runnable demo showing equality with matmul
torch.manual_seed(0)
B, T, d_in, d_out = 2, 3, 4, 5
D = torch.randn(B, T, d_in)            # shape: (B, T, d_in)
A = torch.randn(d_out, d_in)           # shape: (d_out, d_in)

Y1 = einsum(D, A, "... d_in, d_out d_in -> ... d_out")
Y2 = D @ A.T

print('D shape:', tuple(D.shape))
print('A shape:', tuple(A.shape))
print('Y1 (einsum) shape:', tuple(Y1.shape))
print('Y2 (matmul) shape:', tuple(Y2.shape))
print('Allclose:', torch.allclose(Y1, Y2, atol=1e-6))


In [6]:
# Example (einstein_example3): Pixel mixing with einops.rearrange
# Goal: apply a linear transform B over all pixels independently per channel.
# Input: channels_last (batch, height, width, channel); B: (height*width, height*width).

torch.manual_seed(0)
batch, height, width, channel = 64, 32, 32, 3
channels_last = torch.randn(batch, height, width, channel)
B = torch.randn(height * width, height * width)

# 1) Baseline PyTorch (view + transpose + matmul)
channels_last_flat = channels_last.view(batch, height * width, channel)        # (b, hw, c)
channels_first_flat = channels_last_flat.transpose(1, 2)                        # (b, c, hw)
channels_first_flat_transformed = channels_first_flat @ B.T                     # (b, c, hw)
channels_last_flat_transformed = channels_first_flat_transformed.transpose(1, 2) # (b, hw, c)
channels_last_transformed_torch = channels_last_flat_transformed.view(batch, height, width, channel)

# 2) einops.rearrange + einsum (clear, self-documenting dims)
channels_first = rearrange(channels_last, "b h w c -> b c (h w)")
channels_first_transformed = einsum(channels_first, B, "b c pin, pout pin -> b c pout")
channels_last_transformed_einops = rearrange(channels_first_transformed, "b c (h w) -> b h w c", h=height, w=width)

# 3) einx.dot one-liner (einx is like einops.einsum)
channels_last_transformed_einx = einx.dot(
    "batch row_in col_in channel, (row_out col_out) (row_in col_in) -> batch row_out col_out channel",
    channels_last, B, row_in=height, col_in=width, row_out=height, col_out=width,
)

print('channels_last:', tuple(channels_last.shape))
print('B:', tuple(B.shape))
print('torch  ->', tuple(channels_last_transformed_torch.shape))
print('einops ->', tuple(channels_last_transformed_einops.shape))
print('einx   ->', tuple(channels_last_transformed_einx.shape))
print('torch vs einops equal: ', torch.allclose(channels_last_transformed_torch, channels_last_transformed_einops, atol=1e-6))
print('einops vs einx equal:  ', torch.allclose(channels_last_transformed_einops, channels_last_transformed_einx, atol=1e-6))


channels_last: (64, 32, 32, 3)
B: (1024, 1024)
torch  -> (64, 32, 32, 3)
einops -> (64, 32, 32, 3)
einx   -> (64, 32, 32, 3)
torch vs einops equal:  True
einops vs einx equal:   True


In [ ]:
# Toy numeric example + visual demo for dimming
import matplotlib.pyplot as plt

# 1) Tiny numeric example (grayscale 2x2, batch=1, 3 dim levels)
images_small = torch.tensor([[[[1.0],[2.0]], [[3.0],[4.0]]]])  # shape: (1, 2, 2, 1)
dim_by_small = torch.tensor([0.0, 0.5, 1.0])                    # shape: (3,)
dimmed_small = einsum(images_small, dim_by_small, "b h w c, d -> b d h w c")
print('images_small:', images_small.squeeze(-1))
print('dim_by_small:', dim_by_small)
print('dimmed_small (b=0,d=1,0.5x):', dimmed_small[0,1,...].squeeze(-1))
# This shows out[0,1,h,w,0] = images_small[0,h,w,0] * 0.5

# 2) Visual demo on a simple RGB gradient image (H=W=64)
H = W = 64
row = torch.linspace(0, 1, W).unsqueeze(0).expand(H, W)
col = torch.linspace(0, 1, H).unsqueeze(1).expand(H, W)
img = torch.stack([col, row, torch.zeros_like(row)], dim=-1)  # red-green gradient, shape (H, W, 3)
images = img.unsqueeze(0)  # (1, H, W, 3)
dim_by = torch.tensor([0.0, 0.25, 0.5, 0.75, 1.0])
dimmed = einsum(images, dim_by, "b h w c, d -> b d h w c")  # (1, D, H, W, 3)

D = dim_by.numel()
fig, axes = plt.subplots(1, D, figsize=(2.6*D, 2.6))
for i in range(D):
    ax = axes[i] if D > 1 else axes
    ax.imshow(dimmed[0, i].clamp(0, 1).detach().cpu().numpy())
    ax.set_title(f'x {dim_by[i].item():.2f}')
    ax.axis('off')
plt.tight_layout()
plt.show()
